# P02 — Prédiction de la gravité des accidents de la route

**Contexte métier.** ONISR, collectivités et assureurs : anticiper la gravité d'un accident à partir des conditions (usager, véhicule, environnement routier, météo) pour prioriser les secours et cibler la prévention. Enjeu : 3 432 usagers tués en 2024.

**Objectif de l'EDA.** Identifier la cible (gravité de l'usager) et les régresseurs discriminants (âge, catégorie d'usager, éclairage, atmosphère, type de route, surface).

**Données.** Bases de données annuelles des accidents corporels de la circulation 2024 — Ministère de l'Intérieur, data.gouv.fr. Licence ouverte Etalab 2.0. Quatre fichiers : usagers, véhicules, lieux, caractéristiques.

## 1. Téléchargement

Quatre fichiers de l'année 2024 sur data.gouv.fr :

In [ ]:
import urllib.request, os
os.makedirs("data", exist_ok=True)

BASE = "https://static.data.gouv.fr/resources/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024"
FICHIERS = {
    "caract-2024.csv": "20251021-115900",
    "lieux-2024.csv": "20251021-115812",
    "vehicules-2024.csv": "20251107-100240",
    "usagers-2024.csv": "20251021-115506",
}
for nom, stamp in FICHIERS.items():
    if not os.path.exists(f"data/{nom}"):
        urllib.request.urlretrieve(f"{BASE}/{stamp}/{nom}", f"data/{nom}")
        print(nom, "ok")

## 2. Chargement et jointure

Séparateur `;`, décimales à virgule. La cible est portée par le fichier usagers ; on y joint lieux (environnement) et caractéristiques (heure, météo, éclairage).

In [ ]:
import pandas as pd, numpy as np

usagers = pd.read_csv("data/usagers-2024.csv", sep=";", low_memory=False)
lieux = pd.read_csv("data/lieux-2024.csv", sep=";", low_memory=False)
caract = pd.read_csv("data/caract-2024.csv", sep=";", low_memory=False, encoding="latin-1")
veh = pd.read_csv("data/vehicules-2024.csv", sep=";", low_memory=False)

df = usagers.merge(lieux, on="Num_Acc", how="left").merge(caract, on="Num_Acc", how="left")
df["age"] = 2024 - pd.to_numeric(df["an_nais"], errors="coerce")
# grav : 1=Indemne, 2=Tué, 3=Blessé hospitalisé, 4=Blessé léger
df["grave"] = df["grav"].isin([2, 3]).astype(int)
print(len(usagers), "usagers,", df["Num_Acc"].nunique(), "accidents")
print(df["grav"].value_counts().to_dict())

## 3. Analyse

Taux de gravité (tué ou blessé hospitalisé) par tranches d'âge, catégorie d'usager, éclairage, météo, type de route et surface.

In [ ]:
def taux(sub): return round(100 * sub.mean(), 1)

print("Gravité globale :", taux(df["grave"]), "%")
print()
print("Par tranche d'âge :", df.groupby(pd.cut(df["age"], [0, 17, 24, 34, 49, 64, 120]), observed=True)["grave"].mean().mul(100).round(1).to_dict())
print("Par catégorie d'usager (catu) :", df.groupby("catu")["grave"].mean().mul(100).round(1).to_dict())
print("Par éclairage (lum) :", df.groupby("lum")["grave"].mean().mul(100).round(1).to_dict())
print("Par atmosphère (atm) :", df.groupby("atm")["grave"].mean().mul(100).round(1).to_dict())
print("Par type de route (catr) :", df.groupby("catr")["grave"].mean().mul(100).round(1).to_dict())
print("Par surface (surf) :", df.groupby("surf")["grave"].mean().mul(100).round(1).to_dict())
print("Par équipement sécurité (secu1) :", df.groupby("secu1")["grave"].mean().mul(100).round(1).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n usagers : 125187
- n accidents : 54402
- n vehicules : 92678
- grav col : grav (1=Indemne, 2=Tué, 3=Blessé hospitalisé, 4=Blessé léger)
- grav pct : Indemne : 42.27, Blessé léger : 39.71, Blessé hospitalisé : 15.28, Tué : 2.74
- grav rate par catu : 1 : 15.90, 2 : 15.50, 3 : 30.70
- grav rate par classe age : (0, 17] : 19.20, (17, 24] : 17.20, (24, 34] : 14.00, (34, 49] : 13.70, (49, 64] : 17.90, (64, 120] : 27.50
- grav rate par lum : 1 : 16.40, 2 : 19.10, 3 : 27.20, 4 : 20.50, 5 : 12.00
- grav rate par atm : 1 : 17.20, 2 : 13.40, 3 : 16.90, 4 : 15.40, 5 : 26.40, 6 : 29.80, 7 : 26.60, 8 : 15.90, 9 : 25.50
- grav rate par catr : 1 : 10.50, 2 : 19.00, 3 : 24.40, 4 : 11.40, 5 : 24.40, 6 : 20.30, 7 : 15.90, 9 : 26.70
- grav rate par surf : -1 : 34.10, 1 : 17.20, 2 : 15.40, 3 : 12.50, 4 : 21.40, 5 : 14.10, 6 : 44.40, 7 : 19.10, 8 : 26.10, 9 : 34.00
- grav rate par secu1 : 7 : 100.00, 6 : 65.00, 5 : 46.90, 4 : 46.30, 0 : 35.30, 2 : 33.70, 9 : 31.70, 8 : 14.70

## 5. Variable cible et régresseurs

**Cible (classification binaire) : `grave` = 1 si `grav` ∈ {2 Tué, 3 Blessé hospitalisé}.** La classe positive est minoritaire (~18 %) : problème de classification déséquilibrée.

**Régresseurs retenus :**
- `age` (calculé depuis `an_nais`) : gravité 27,5 % chez les 65+ contre 13,7 % chez les 35-49 ans
- `catu` (piéton/cycliste/occupant) : piétons ~30,7 % de gravité
- `lum` (éclairage) : gravité doublée de nuit ; `atm` (météo) : pluie/grêle aggravantes
- `catr` (type de route) : départementales et nationales plus meurtrières qu'autoroutes
- `surf` (état de la surface), `secu1` (équipement de sécurité)
- Variables véhicule : `catv`, `obsm` (obstacle mobile heurté)

**Pièges :**
- Codage grav : 1=Indemne, 2=Tué, 3=Blessé hospitalisé, 4=Blessé léger — l'ordre n'est pas monotone, ne pas traiter comme ordinal naïvement
- Ne jamais utiliser des variables connues après l'accident (choc, nombre de blessés)
- La cible est par usager : plusieurs lignes par accident → corrélation intra-accident à gérer (échantillonnage ou features agrégées)
- Encodage latin-1 du fichier caractéristiques